# Decision Trees and Ensemble Methods

This notebook covers:
1. **Decision Trees** -- interpretable, non-linear models
2. **Random Forest** -- bagging of decision trees
3. **Gradient Boosting / XGBoost** -- sequential ensemble learning

We compare their performance on a classification task.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not installed -- pip install xgboost')

%matplotlib inline

In [ ]:
# Load Wine dataset
wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 1. Decision Tree

A decision tree recursively splits the feature space by choosing the feature and threshold that maximise **information gain** (or minimise Gini impurity).

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
print(f"Decision Tree accuracy: {accuracy_score(y_test, dt.predict(X_test)):.3f}")

# Visualise the tree
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(dt, feature_names=wine.feature_names, class_names=wine.target_names,
          filled=True, rounded=True, ax=ax, fontsize=8)
plt.title('Decision Tree (max_depth=4)')
plt.tight_layout()
plt.show()

## 2. Random Forest

Random Forest trains $B$ independent trees on **bootstrap samples** and averages predictions. Each split considers only a random subset of $\sqrt{p}$ features, reducing correlation between trees.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42)
rf.fit(X_train, y_train)
print(f"Random Forest accuracy: {accuracy_score(y_test, rf.predict(X_test)):.3f}")

# Feature importance
importances = pd.Series(rf.feature_importances_, index=wine.feature_names)
importances.sort_values().plot.barh()
plt.title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()

## 3. Gradient Boosting and XGBoost

Boosting builds trees **sequentially**: each new tree fits the residual errors of the ensemble so far.
$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$
where $\eta$ is the learning rate.

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                 max_depth=3, random_state=42)
gb.fit(X_train, y_train)
print(f"GradientBoosting accuracy: {accuracy_score(y_test, gb.predict(X_test)):.3f}")

if HAS_XGB:
    xgb = XGBClassifier(n_estimators=200, learning_rate=0.1,
                         max_depth=3, use_label_encoder=False,
                         eval_metric='mlogloss', random_state=42)
    xgb.fit(X_train, y_train)
    print(f"XGBoost accuracy:         {accuracy_score(y_test, xgb.predict(X_test)):.3f}")

In [ ]:
# Compare models via cross-validation
models = {'Decision Tree': dt, 'Random Forest': rf, 'GradientBoosting': gb}
if HAS_XGB:
    models['XGBoost'] = xgb

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    results[name] = scores
    print(f"{name:20s} CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

plt.boxplot(results.values(), labels=results.keys())
plt.ylabel('Accuracy')
plt.title('5-Fold Cross-Validation Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Key Takeaways

- **Decision trees** are interpretable but prone to overfitting.
- **Random Forest** reduces variance through bagging and feature subsampling.
- **Gradient Boosting / XGBoost** reduces bias through sequential residual fitting.
- In practice, ensembles nearly always outperform single trees.

**Next:** Unsupervised learning (clustering, dimensionality reduction).